# Caravan Insurance Classification with K-Nearest Neighbours

This project adapts and expands coursework I completed through the University of Toronto Data Sciences Institute. The original assignment introduced K-nearest neighbours (KNN), feature scaling, train/test splitting and model evaluation using the `Caravan` dataset from the `ISLP` package.

For this standalone portfolio version, I reorganized the analysis and tightened the workflow by using a reproducible stratified split, fitting preprocessing only on the training data, and adding class-aware evaluation metrics.


## 1. Imports and data

The dataset contains customer characteristics and a binary target, `Purchase`, indicating whether a customer purchased caravan insurance.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ISLP import load_data
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

caravan = load_data("Caravan")
caravan.head()


## 2. Dataset overview

The original coursework used 5,822 observations, 85 predictor variables and one response variable. The positive class is rare, so accuracy alone can be misleading.


In [ ]:
print("Shape:", caravan.shape)
print("Predictors:", caravan.shape[1] - 1)

purchase_rate = caravan["Purchase"].eq("Yes").mean()
print(f"Purchase rate: {purchase_rate:.2%}")

caravan["Purchase"].value_counts()


Because only about 6% of customers purchased insurance in the original dataset, a model can achieve high raw accuracy simply by predicting `No` most of the time. I therefore compare KNN to a majority-class baseline and report balanced accuracy, precision, recall and F1 in addition to accuracy.


## 3. Reproducible train/test split

The target is separated from the predictors and the data is split 75/25. `stratify=y` preserves the class distribution in both subsets.


In [ ]:
X = caravan.drop(columns="Purchase")
y = caravan["Purchase"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print(f"Training purchase rate: {y_train.mean():.2%}")
print(f"Testing purchase rate: {y_test.mean():.2%}")


## 4. KNN pipeline

KNN is distance-based, so the predictor variables need to be on comparable scales. A scikit-learn `Pipeline` fits `StandardScaler` on the training data only and then applies the same transformation to the test data.


In [ ]:
def make_knn(k):
    return Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ])

knn_1 = make_knn(1)
knn_3 = make_knn(3)

knn_1.fit(X_train, y_train)
knn_3.fit(X_train, y_train)


## 5. Evaluation


In [ ]:
def evaluate(model, X_test, y_test, label):
    pred = model.predict(X_test)
    return {
        "model": label,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    }

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

results = pd.DataFrame([
    evaluate(baseline, X_test, y_test, "Majority baseline"),
    evaluate(knn_1, X_test, y_test, "KNN (k=1)"),
    evaluate(knn_3, X_test, y_test, "KNN (k=3)"),
]).set_index("model")

results.round(3)


A useful model should be judged against the class imbalance rather than raw accuracy alone. The table above makes that comparison explicit.


In [ ]:
pred_3 = knn_3.predict(X_test)
print("Confusion matrix for KNN (k=3):")
print(confusion_matrix(y_test, pred_3))
print("\nClassification report:")
print(classification_report(y_test, pred_3, target_names=["No", "Yes"], zero_division=0))


## 6. Small sensitivity check for k

The original coursework compared `k=1` and `k=3`. Here I also check a few nearby values to see how sensitive performance is to the neighbourhood size.


In [ ]:
rows = []
for k in [1, 3, 5, 7, 9]:
    model = make_knn(k)
    model.fit(X_train, y_train)
    rows.append(evaluate(model, X_test, y_test, f"k={k}"))

k_results = pd.DataFrame(rows).set_index("model")
k_results.round(3)


## Takeaways

- KNN requires feature scaling because distance calculations are sensitive to predictor scale.
- The response variable should not be standardized; it is the class label the model is trying to predict.
- A stratified split makes the comparison reproducible while preserving the rare positive class.
- With a strongly imbalanced target, raw accuracy can look impressive even when the model identifies few purchasers.
- Balanced accuracy, precision, recall, F1 and the confusion matrix provide a more useful view of model behaviour.

### Provenance

This project is adapted from an assignment I completed in the University of Toronto Data Sciences Institute Machine Learning Software Foundations program. The portfolio version reorganizes and expands that work for clarity and reproducibility.
